# 04 - Evaluation & Comparison

**Goal:** Evaluate 3 FL-trained models (GCN, GAT, GraphSAGE) on the test graph. Compare metrics.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (14, 5),
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.dpi': 200,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight'
})
sns.set_style('whitegrid')

OUT = Path('D:/Project/riset/fl-gnn/dataset-clean')
RESULTS = Path('D:/Project/riset/fl-gnn/results')
EVAL_DIR = Path('D:/Project/riset/fl-gnn/evaluation_results')
EVAL_DIR.mkdir(exist_ok=True)

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'CUDA: {torch.cuda.get_device_name(0)}')
else:
    device = torch.device('cpu')
    print('CPU mode')
print('Libraries ready.')

In [ ]:
HIDDEN_DIM = 256
DROPOUT = 0.3
MODELS = ['GCN', 'GAT', 'GraphSAGE']
USE_AMP = False

## 1. Load Data & Models

In [ ]:
print('Loading test graph...')
data = np.load(OUT / 'test_graph.npz')
X = torch.from_numpy(data['X']).to(device)
y = torch.from_numpy(data['y']).long().to(device)
edge_index = torch.from_numpy(data['edge_index']).to(device)
in_dim = X.shape[1]
num_classes = len(np.unique(y.cpu().numpy()))
print(f'Test graph: {X.shape[0]} nodes, {edge_index.shape[1]} edges, {num_classes} classes')

In [ ]:
from torch_geometric.nn import GCNConv, GATConv, SAGEConv

class GCNModel(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.conv3 = GCNConv(hidden, out_dim)
        self.dropout = dropout
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.conv2(x, edge_index).relu()
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.conv3(x, edge_index)
        return x

class GATModel(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden // 4, heads=4, concat=True)
        self.conv2 = GATConv(hidden, hidden // 4, heads=4, concat=True)
        self.conv3 = GATConv(hidden, out_dim, heads=1, concat=False)
        self.dropout = dropout
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.conv2(x, edge_index).relu()
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.conv3(x, edge_index)
        return x

class SAGEModel(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.conv3 = SAGEConv(hidden, out_dim)
        self.dropout = dropout
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.conv2(x, edge_index).relu()
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.conv3(x, edge_index)
        return x

MODEL_CLASSES = {'GCN': GCNModel, 'GAT': GATModel, 'GraphSAGE': SAGEModel}

print('Loading trained models...')
models = {}
for arch in MODELS:
    path = RESULTS / f'{arch}_global_model.pt'
    if path.exists():
        model = MODEL_CLASSES[arch](in_dim, HIDDEN_DIM, num_classes, DROPOUT).to(device)
        model.load_state_dict(torch.load(path, map_location=device))
        models[arch] = model
        print(f'  {arch}: loaded ({sum(p.numel() for p in model.parameters()):,} params)')
    else:
        print(f'  {arch}: NOT FOUND ({path})')

## 2. Evaluate All Models

In [ ]:
results = {}
all_preds = {}
all_probs = {}

for arch, model in models.items():
    print(f'\nEvaluating {arch}...')
    model.eval()

    with torch.no_grad(), torch.amp.autocast('cuda', enabled=USE_AMP):
        out = model(X, edge_index)
        loss = F.cross_entropy(out, y).item()
        probs = F.softmax(out, dim=1)
        preds = out.argmax(dim=1).cpu().numpy()

    y_true = y.cpu().numpy()
    acc = accuracy_score(y_true, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, preds, average='weighted', zero_division=0
    )
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, preds, average='macro', zero_division=0
    )

    results[arch] = {
        'loss': loss,
        'accuracy': acc,
        'precision_weighted': precision,
        'recall_weighted': recall,
        'f1_weighted': f1,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'predictions': preds,
    }
    all_preds[arch] = preds
    all_probs[arch] = probs.cpu().numpy()

    print(f'  Acc: {acc:.4f}')
    print(f'  F1 (weighted): {f1:.4f}')
    print(f'  F1 (macro):    {f1_macro:.4f}')
    print(f'  Loss: {loss:.4f}')

In [ ]:
df = pd.DataFrame(results).T
display_cols = ['accuracy', 'f1_weighted', 'f1_macro', 'precision_weighted',
                'recall_weighted', 'loss']
print(df[display_cols].round(4).to_string())

df[display_cols].round(4).to_csv(EVAL_DIR / 'metrics_comparison.csv')
print('\nSaved to evaluation_results/metrics_comparison.csv')

## 3. Confusion Matrix

In [ ]:
import joblib
le = joblib.load(OUT / 'label_encoder.pkl')
class_names = le.classes_

fig, axes = plt.subplots(1, len(models), figsize=(7 * len(models), 6))
if len(models) == 1:
    axes = [axes]

for idx, (arch, model) in enumerate(models.items()):
    cm = confusion_matrix(y.cpu().numpy(), all_preds[arch])

    # Top-10 most frequent classes in test
    class_counts = np.bincount(y.cpu().numpy())
    top10 = np.argsort(class_counts)[-10:]

    im = axes[idx].imshow(cm[top10[:, None], top10], cmap='Blues', aspect='auto')
    axes[idx].set_xticks(range(10))
    axes[idx].set_yticks(range(10))
    axes[idx].set_xticklabels([class_names[t] for t in top10], rotation=45, fontsize=7)
    axes[idx].set_yticklabels([class_names[t] for t in top10], fontsize=7)
    axes[idx].set_title(f'{arch} (top-10 classes)')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('True')
    plt.colorbar(im, ax=axes[idx])

plt.tight_layout()
plt.savefig(EVAL_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Per-Class F1

In [ ]:
y_true = y.cpu().numpy()

f1_per_class = {}
for arch in MODELS:
    if arch in all_preds:
        _, _, f1, _ = precision_recall_fscore_support(
            y_true, all_preds[arch], average=None, zero_division=0
        )
        f1_per_class[arch] = f1

nz_classes = []
nz_f1 = {arch: [] for arch in f1_per_class}
for c in range(num_classes):
    if any(f1_per_class[arch][c] > 0 for arch in f1_per_class):
        nz_classes.append(class_names[c] if c < len(class_names) else f'Class_{c}')
        for arch in f1_per_class:
            nz_f1[arch].append(f1_per_class[arch][c])

x = np.arange(len(nz_classes))
w = 0.25
plt.figure(figsize=(20, 8))
for i, arch in enumerate(f1_per_class):
    plt.bar(x + i * w, nz_f1[arch], w, label=arch, alpha=0.8)
plt.xticks(x + w, nz_classes, rotation=90, fontsize=7)
plt.ylabel('F1 Score')
plt.xlabel('Class')
plt.title('Per-Class F1 Score Comparison')
plt.legend()
plt.ylim(0, 1.05)
plt.tight_layout()
plt.savefig(EVAL_DIR / 'per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()

f1_df = pd.DataFrame(nz_f1, index=nz_classes)
f1_df.to_csv(EVAL_DIR / 'per_class_f1.csv')
print('Per-class F1 saved.')

In [ ]:
print('='*60)
print('FINAL COMPARISON')
print('='*60)
print(f'{"Model":<12} {"Accuracy":<10} {"F1 (w)":<10} {"F1 (macro)":<12} {"Loss":<10}')
print('-'*54)
for arch in MODELS:
    if arch in results:
        r = results[arch]
        print(f'{arch:<12} {r["accuracy"]:<10.4f} {r["f1_weighted"]:<10.4f} '
              f'{r["f1_macro"]:<12.4f} {r["loss"]:<10.4f}')
print('='*60)